#### NCSU GIS 714: Geospatial Computation and Simulations

## Spatio-temporal data handling and visualization in GRASS GIS

Additional Resources:
* [GRASS GIS Overview and Manual](https://grass.osgeo.org/grass-stable/manuals/index.html)
* [GRASS book](http://www.grassbook.org/)

## Climate Data Analysis

### Start GRASS GIS

In [1]:
import subprocess
import sys
import json
import matplotlib.pyplot as plt
from datetime import datetime

# Ask GRASS GIS where its Python packages are.
grass_call = "grass84"
shell_setting = True

sys.path.append(
    subprocess.check_output([grass_call, "--config", "python_path"], text=True, shell=shell_setting).strip()
)

# Import GRASS packages
import grass.script as gs
import grass.jupyter as gj

# Start GRASS Session
gj.init("C:\\Users\\Nietolaj\\OneDrive - Mondigroup\\NC State\\Spring 2026\\Geospatial Computation and Simulation\\GIS714-assignments-main\\GIS714-assignments-main\\data\\", "Lapse_Rate")

Computational Region options.

In [2]:
#Mondi South Africa
gs.run_command("g.region", raster=["Zululand_DEM@DEMs", "Midlands_DEM@DEMs", "Central_DEM@DEMs"])

#Zululand
#gs.run_command("g.region", raster="Zululand_DEM@DEMs")

#Midlands
#gs.run_command("g.region", raster="Midlands_DEM@DEMs")

#Central
#gs.run_command("g.region", raster="Central_DEM@DEMs")

Check available mapsets

In [3]:
print(gs.read_command("g.mapset", flags="l"))

DEMs PERMANENT Temperature 



List available raster maps. 

In [4]:
print(gs.read_command("g.list", type="raster", mapset="DEMs"))

Central_DEM
Midlands_DEM
Zululand_DEM



Import DEMs if necessary. This only ever needs to be run once.

In [5]:
gs.run_command("g.mapset", mapset="DEMs")
gs.run_command("g.region", raster=["Zululand_DEM@DEMs", "Midlands_DEM@DEMs", "Central_DEM@DEMs"])

#Zululand.
#gs.run_command("r.in.gdal", input=r'OpenFileGDB:"C:\Users\Nietolaj\OneDrive - Mondigroup\Documents\ArcGIS\Projects\Erosion Layers\Erosion Layers.gdb":Fill_Zululand', output="Zululand_DEM")

#Midlands.
#gs.run_command("r.in.gdal", input=r'OpenFileGDB:"C:\Users\Nietolaj\OneDrive - Mondigroup\Documents\ArcGIS\Projects\Erosion Layers\Erosion Layers.gdb":Fill_Midlands', output="Midlands_DEM")

#Central.
#gs.run_command("r.in.gdal", input=r'OpenFileGDB:"C:\Users\Nietolaj\OneDrive - Mondigroup\Documents\ArcGIS\Projects\Erosion Layers\Erosion Layers.gdb":Fill_CENTRAL', output="Central_DEM")

#Joining all rasters into one.
#gs.run_command("r.patch", input=["Zululand_DEM", "Midlands_DEM", "Central_DEM"], output="Mondi_DEM")

Convert plantation vector into a raster, and then create a coefficient raster. This only ever needs to be run once.

In [8]:
gs.run_command("g.mapset", mapset="PERMANENT")
gs.run_command("g.region", raster=["Zululand_DEM@DEMs", "Midlands_DEM@DEMs", "Central_DEM@DEMs"])

#gs.run_command("v.to.rast", input="Plantations", output="AWSID", use="attr", attribute_column="AWS_ID")
#gs.run_command("r.recode", input="AWSID", output="TMAX_Coefficients", rules="tmax_rules.txt")
gs.run_command("r.recode", input="AWSID", output="TMIN_Coefficients", rules="tmin_rules.txt")

### Applying lapse rate

Establish base raster.

In [7]:
gs.run_command("g.mapset", mapset="Temperature")
gs.run_command("g.region", raster="Midlands_DEM@DEMs")
#gs.run_command("g.region", raster="Zululand_DEM@DEMs")
#gs.run_command("g.region", raster="Central_DEM@DEMs")

#This takes about 30 minutes.
gs.run_command("r.mapcalc", expression ="Base_TMAX = 1.958588 + (-0.002886)*Midlands_DEM@DEMs + 0.997383*TMAX_08032022")
#gs.run_command("r.mapcalc", expression ="Base_TMIN = -1.486394 + (0.0008948)*Midlands_DEM@DEMs + 1.050171*TMIN_08032022")

Adjust by weather station.

In [42]:
#This takes about 20 minutes.
gs.run_command("r.mapcalc", expression ="TMAX_final = Base_TMAX + TMAX_Coefficients@PERMANENT")
#gs.run_command("r.mapcalc", expression ="TMAX_final = Base_TMIN + TMIN_Coefficients@PERMANENT")

In [ ]:
print("skr")